# Instalasi dan Verifikasi Dependensi

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import torchvision
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import matplotlib.pyplot as plt
import time

# Cek versi PyTorch
print(f"PyTorch Version : {torch.__version__}")
print(f"Torchvision     : {torchvision.__version__}")

# Cek ketersediaan GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device          : {device}")

In [ ]:
# Contoh struktur Generator (vanilla GAN) dengan nn.Sequential
generator = nn.Sequential(
    nn.Linear(100, 256),        # Noise (100) → 256
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Linear(256, 784),        # 256 → 784 (28×28)
    nn.Tanh()                   # Output: [-1, 1]
)

# Contoh struktur Discriminator (vanilla GAN) dengan nn.Sequential
discriminator = nn.Sequential(
    nn.Linear(784, 256),        # Gambar (784) → 256
    nn.LeakyReLU(0.2),
    nn.Linear(256, 1),          # 256 → 1
    nn.Sigmoid()                # Output: probabilitas [0, 1]
)

In [ ]:
'''
nn.ConvTranspose2d(
    in_channels,     # Jumlah channel input
    out_channels,    # Jumlah channel output
    kernel_size,     # Ukuran kernel
    stride=1,        # Stride (stride > 1 → upsampling)
    padding=0,       # Padding
    output_padding=0 # Padding tambahan pada output
)
'''

In [ ]:
# ============================
# GENERATOR: NOISE → DATA
# ============================

torch.manual_seed(42)

# Generator paling sederhana: noise 2D → output 2D
# Belum dilatih — hanya untuk memahami struktur

generator_demo = nn.Sequential(
    nn.Linear(2, 16),       # Input: noise 2D
    nn.ReLU(),
    nn.Linear(16, 32),      # Hidden layer
    nn.ReLU(),
    nn.Linear(32, 2)        # Output: data 2D (belum pakai Tanh)
)

print("Generator (Demo):")
print(generator_demo)

# Generate data palsu dari noise
noise = torch.randn(500, 2)  # 500 sampel noise dari N(0,1)
print(f"\nNoise shape: {noise.shape}")
print(f"Noise statistik: mean={noise.mean():.3f}, std={noise.std():.3f}")

# Forward pass melalui generator (BELUM dilatih)
with torch.no_grad():
    fake_data = generator_demo(noise)

print(f"Fake data shape: {fake_data.shape}")

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(noise[:, 0].numpy(), noise[:, 1].numpy(),
                alpha=0.4, s=10, color='steelblue')
axes[0].set_title('Input: Noise z ~ N(0,1)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('z₁', fontsize=12)
axes[0].set_ylabel('z₂', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-4, 4)
axes[0].set_ylim(-4, 4)

axes[1].scatter(fake_data[:, 0].numpy(), fake_data[:, 1].numpy(),
                alpha=0.4, s=10, color='coral')
axes[1].set_title('Output: G(z) — Belum Dilatih', fontsize=13, fontweight='bold')
axes[1].set_xlabel('x₁', fontsize=12)
axes[1].set_ylabel('x₂', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Generator Mengubah Noise Menjadi Data',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# DISCRIMINATOR: DATA → PROBABILITAS ASLI/PALSU
# ============================

# Discriminator: menerima data 2D, output probabilitas "asli" (0-1)

discriminator_demo = nn.Sequential(
    nn.Linear(2, 16),
    nn.LeakyReLU(0.2),
    nn.Linear(16, 32),
    nn.LeakyReLU(0.2),
    nn.Linear(32, 1),
    nn.Sigmoid()            # Output: probabilitas [0, 1]
)

print("Discriminator (Demo):")
print(discriminator_demo)

# Buat data "asli" (lingkaran)
theta = torch.linspace(0, 2 * np.pi, 500)
radius = 2 + torch.randn(500) * 0.1
real_data = torch.stack([radius * torch.cos(theta),
                          radius * torch.sin(theta)], dim=1)

# Data "palsu" dari Generator (yang belum dilatih)
with torch.no_grad():
    noise = torch.randn(500, 2)
    fake_data = generator_demo(noise)

# Forward pass Discriminator (BELUM dilatih)
with torch.no_grad():
    real_scores = discriminator_demo(real_data)    # Skor untuk data asli
    fake_scores = discriminator_demo(fake_data)    # Skor untuk data palsu

print(f"\nSkor Discriminator (BELUM dilatih):")
print(f"  Data ASLI  — mean: {real_scores.mean():.4f}, "
      f"min: {real_scores.min():.4f}, max: {real_scores.max():.4f}")
print(f"  Data PALSU — mean: {fake_scores.mean():.4f}, "
      f"min: {fake_scores.min():.4f}, max: {fake_scores.max():.4f}")
print(f"\n  → Discriminator belum dilatih, jadi skor-nya hampir sama (acak ~0.5)")

In [ ]:
# ============================
# VISUALISASI: DATA ASLI vs PALSU vs SKOR DISCRIMINATOR
# ============================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Data asli
axes[0].scatter(real_data[:, 0].numpy(), real_data[:, 1].numpy(),
                alpha=0.5, s=10, color='steelblue', label='Data Asli')
axes[0].set_title('Data Asli (Lingkaran)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('x₁', fontsize=12)
axes[0].set_ylabel('x₂', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

# Plot 2: Data palsu
axes[1].scatter(fake_data[:, 0].numpy(), fake_data[:, 1].numpy(),
                alpha=0.5, s=10, color='coral', label='Data Palsu G(z)')
axes[1].set_title('Data Palsu — G(z)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('x₁', fontsize=12)
axes[1].set_ylabel('x₂', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

# Plot 3: Keduanya + skor
axes[2].scatter(real_data[:, 0].numpy(), real_data[:, 1].numpy(),
                alpha=0.4, s=10, color='steelblue', label='Asli')
axes[2].scatter(fake_data[:, 0].numpy(), fake_data[:, 1].numpy(),
                alpha=0.4, s=10, color='coral', label='Palsu')
axes[2].set_title('Asli vs Palsu (Belum Dilatih)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('x₁', fontsize=12)
axes[2].set_ylabel('x₂', fontsize=12)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)
axes[2].set_aspect('equal')

plt.suptitle('Generator & Discriminator — Sebelum Training',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# BCELoss UNTUK GAN
# ============================

criterion = nn.BCELoss()

# Labels
real_labels = torch.ones(5, 1)   # Label 1 = "asli"
fake_labels = torch.zeros(5, 1)  # Label 0 = "palsu"

print("=== SKENARIO DISCRIMINATOR ===\n")

# Skenario 1: Discriminator SEMPURNA
perfect_real_pred = torch.tensor([[0.99], [0.98], [0.97], [0.99], [0.95]])
perfect_fake_pred = torch.tensor([[0.01], [0.02], [0.03], [0.01], [0.05]])

loss_real = criterion(perfect_real_pred, real_labels)
loss_fake = criterion(perfect_fake_pred, fake_labels)
print(f"Discriminator SEMPURNA:")
print(f"  Loss data asli (D(x)≈1)  : {loss_real.item():.4f} (rendah (V))")
print(f"  Loss data palsu (D(G(z))≈0): {loss_fake.item():.4f} (rendah (V))")
print(f"  Total loss D              : {(loss_real + loss_fake).item():.4f}")

# Skenario 2: Discriminator BURUK — tidak bisa bedakan
bad_real_pred = torch.tensor([[0.50], [0.48], [0.52], [0.51], [0.49]])
bad_fake_pred = torch.tensor([[0.50], [0.52], [0.48], [0.49], [0.51]])

loss_real = criterion(bad_real_pred, real_labels)
loss_fake = criterion(bad_fake_pred, fake_labels)
print(f"\nDiscriminator BURUK (menebak acak):")
print(f"  Loss data asli (D(x)≈0.5) : {loss_real.item():.4f} (tinggi (x))")
print(f"  Loss data palsu (D(G(z))≈0.5): {loss_fake.item():.4f} (tinggi (x))")
print(f"  Total loss D               : {(loss_real + loss_fake).item():.4f}")

print("\n=== SKENARIO GENERATOR ===\n")

# Skenario: Generator BERHASIL menipu Discriminator
# Loss G = BCE(D(G(z)), label=1) — G ingin D(G(z)) → 1
gen_fooled = torch.tensor([[0.95], [0.92], [0.88], [0.91], [0.94]])
gen_failed = torch.tensor([[0.10], [0.15], [0.08], [0.12], [0.05]])

loss_g_good = criterion(gen_fooled, real_labels)
loss_g_bad = criterion(gen_failed, real_labels)
print(f"Generator BERHASIL menipu D:")
print(f"  D(G(z)) ≈ 0.9 → Loss G: {loss_g_good.item():.4f} (rendah (V))")
print(f"\nGenerator GAGAL menipu D:")
print(f"  D(G(z)) ≈ 0.1 → Loss G: {loss_g_bad.item():.4f} (tinggi (x))")

In [ ]:
# ============================
# DATA TARGET: DISTRIBUSI GAUSSIAN
# ============================

torch.manual_seed(42)

# Data target: Gaussian dengan mean=4, std=1.5
real_mean = 4.0
real_std = 1.5
n_samples = 5000

real_data_1d = torch.normal(mean=real_mean, std=real_std, size=(n_samples, 1))

print(f"Data target — Mean: {real_data_1d.mean():.3f}, Std: {real_data_1d.std():.3f}")

# Visualisasi
plt.figure(figsize=(10, 5))
plt.hist(real_data_1d.numpy(), bins=60, density=True, alpha=0.7,
         color='steelblue', edgecolor='white', label='Data Asli')
plt.axvline(real_mean, color='crimson', linewidth=2, linestyle='--',
            label=f'Mean = {real_mean}')
plt.xlabel('Nilai', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title('Distribusi Data Target: N(4.0, 1.5²)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# GENERATOR: noise (1D) → data (1D)
# ============================

latent_dim = 8  # Dimensi noise

generator_1d = nn.Sequential(
    nn.Linear(latent_dim, 32),
    nn.ReLU(),
    nn.Linear(32, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)           # Output 1D, tanpa aktivasi
)

print("Generator 1D:")
print(generator_1d)
print(f"Parameter: {sum(p.numel() for p in generator_1d.parameters()):,}")

# ============================
# DISCRIMINATOR: data (1D) → probabilitas (0-1)
# ============================

discriminator_1d = nn.Sequential(
    nn.Linear(1, 32),
    nn.LeakyReLU(0.2),
    nn.Linear(32, 64),
    nn.LeakyReLU(0.2),
    nn.Linear(64, 32),
    nn.LeakyReLU(0.2),
    nn.Linear(32, 1),
    nn.Sigmoid()               # Probabilitas asli/palsu
)

print("\nDiscriminator 1D:")
print(discriminator_1d)
print(f"Parameter: {sum(p.numel() for p in discriminator_1d.parameters()):,}")

In [ ]:
# ============================
# LOSS FUNCTION & OPTIMIZER
# ============================

criterion = nn.BCELoss()

# PENTING: 2 optimizer terpisah — satu untuk G, satu untuk D!
optimizer_G = optim.Adam(generator_1d.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator_1d.parameters(), lr=0.0002, betas=(0.5, 0.999))

# DataLoader
train_dataset = TensorDataset(real_data_1d)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

# ============================
# TRAINING LOOP GAN
# ============================

num_epochs = 200
d_losses = []
g_losses = []
# Menyimpan distribusi Generator setiap beberapa epoch untuk visualisasi
gen_snapshots = {}

print("Mulai Training GAN 1D...")
print("=" * 65)

for epoch in range(num_epochs):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0
    num_batches = 0

    for (real_batch,) in train_loader:
        batch_size = real_batch.size(0)

        # Label
        real_label = torch.ones(batch_size, 1)
        fake_label = torch.zeros(batch_size, 1)

        # =============================
        # LANGKAH 1: LATIH DISCRIMINATOR
        # =============================

        # 1a. Loss pada data ASLI
        real_output = discriminator_1d(real_batch)
        d_loss_real = criterion(real_output, real_label)

        # 1b. Loss pada data PALSU
        noise = torch.randn(batch_size, latent_dim)
        fake_data = generator_1d(noise)
        fake_output = discriminator_1d(fake_data.detach())  # detach: JANGAN hitung gradien G
        d_loss_fake = criterion(fake_output, fake_label)

        # 1c. Total loss Discriminator
        d_loss = d_loss_real + d_loss_fake

        # 1d. Backprop & update Discriminator
        optimizer_D.zero_grad()
        d_loss.backward()
        optimizer_D.step()

        # =============================
        # LANGKAH 2: LATIH GENERATOR
        # =============================

        # 2a. Generate data palsu baru
        noise = torch.randn(batch_size, latent_dim)
        fake_data = generator_1d(noise)

        # 2b. Loss Generator: ingin D menganggap fake sebagai real
        fake_output = discriminator_1d(fake_data)
        g_loss = criterion(fake_output, real_label)  # Target: 1 (ingin dianggap asli!)

        # 2c. Backprop & update Generator
        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()

        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()
        num_batches += 1

    avg_d_loss = epoch_d_loss / num_batches
    avg_g_loss = epoch_g_loss / num_batches
    d_losses.append(avg_d_loss)
    g_losses.append(avg_g_loss)

    # Simpan snapshot distribusi Generator
    if (epoch + 1) in [1, 10, 50, 100, 150, 200]:
        with torch.no_grad():
            snapshot_noise = torch.randn(5000, latent_dim)
            snapshot_data = generator_1d(snapshot_noise)
            gen_snapshots[epoch + 1] = snapshot_data.numpy().flatten()

    # Print progress
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
              f"D Loss: {avg_d_loss:.4f} | G Loss: {avg_g_loss:.4f}")

print("=" * 65)
print("Training Selesai!")

In [ ]:
# ============================
# PLOT TRAINING LOSS
# ============================

plt.figure(figsize=(12, 5))
plt.plot(d_losses, label='Discriminator Loss', color='steelblue', linewidth=2)
plt.plot(g_losses, label='Generator Loss', color='coral', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss — GAN 1D', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# EVOLUSI DISTRIBUSI GENERATOR SEIRING TRAINING
# ============================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (ep, data) in enumerate(gen_snapshots.items()):
    axes[idx].hist(real_data_1d.numpy().flatten(), bins=60, density=True,
                   alpha=0.5, color='steelblue', edgecolor='white', label='Data Asli')
    axes[idx].hist(data, bins=60, density=True,
                   alpha=0.5, color='coral', edgecolor='white', label='Generator')
    axes[idx].axvline(real_mean, color='green', linewidth=1.5, linestyle='--',
                      alpha=0.7)
    axes[idx].set_title(f'Epoch {ep}', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('Nilai', fontsize=11)
    axes[idx].set_ylabel('Density', fontsize=11)
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Evolusi Distribusi Generator Selama Training',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# EVALUASI AKHIR
# ============================

# Generate data dari Generator yang sudah dilatih
with torch.no_grad():
    final_noise = torch.randn(10000, latent_dim)
    generated_data = generator_1d(final_noise).numpy().flatten()

print("=== PERBANDINGAN STATISTIK ===")
print(f"{'Metrik':<15} | {'Data Asli':<15} | {'Generated':<15}")
print("-" * 50)
print(f"{'Mean':<15} | {real_data_1d.mean().item():<15.4f} | {generated_data.mean():<15.4f}")
print(f"{'Std':<15} | {real_data_1d.std().item():<15.4f} | {generated_data.std():<15.4f}")
print(f"{'Min':<15} | {real_data_1d.min().item():<15.4f} | {generated_data.min():<15.4f}")
print(f"{'Max':<15} | {real_data_1d.max().item():<15.4f} | {generated_data.max():<15.4f}")

# Visualisasi final
plt.figure(figsize=(12, 5))
plt.hist(real_data_1d.numpy().flatten(), bins=80, density=True,
         alpha=0.5, color='steelblue', edgecolor='white', label='Data Asli N(4, 1.5²)')
plt.hist(generated_data, bins=80, density=True,
         alpha=0.5, color='coral', edgecolor='white', label='Generator (setelah training)')
plt.xlabel('Nilai', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title('GAN 1D — Data Asli vs Generated', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# DATASET MNIST
# ============================

torch.manual_seed(42)

# Transform: konversi ke tensor & normalisasi ke [-1, 1] (untuk Tanh output)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalisasi ke [-1, 1]
])

# Download dataset
train_dataset = datasets.MNIST(root='./data', train=True,
                                download=True, transform=transform)

# DataLoader
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

print(f"Training samples : {len(train_dataset)}")
print(f"Image shape      : {train_dataset[0][0].shape}")   # (1, 28, 28)
print(f"Batch size       : {batch_size}")
print(f"Training batches : {len(train_loader)}")

# Visualisasi beberapa sampel
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    # Denormalisasi dari [-1,1] ke [0,1] untuk visualisasi
    ax.imshow(image.squeeze() * 0.5 + 0.5, cmap='gray')
    ax.set_title(f'{label}', fontsize=10)
    ax.axis('off')

plt.suptitle('Sampel MNIST — Data Asli', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# GENERATOR: noise (100) → gambar (1×28×28 = 784)
# ============================

latent_dim = 100  # Dimensi latent space
img_dim = 28 * 28  # 784 piksel

generator_mnist = nn.Sequential(
    # Layer 1: 100 → 256
    nn.Linear(latent_dim, 256),
    nn.BatchNorm1d(256),
    nn.LeakyReLU(0.2),

    # Layer 2: 256 → 512
    nn.Linear(256, 512),
    nn.BatchNorm1d(512),
    nn.LeakyReLU(0.2),

    # Layer 3: 512 → 1024
    nn.Linear(512, 1024),
    nn.BatchNorm1d(1024),
    nn.LeakyReLU(0.2),

    # Layer 4: 1024 → 784 (28×28)
    nn.Linear(1024, img_dim),
    nn.Tanh()                   # Output: [-1, 1]
)

print("Generator MNIST:")
print(generator_mnist)
print(f"\nTotal parameter Generator: {sum(p.numel() for p in generator_mnist.parameters()):,}")

In [ ]:
# ============================
# DISCRIMINATOR: gambar (784) → probabilitas (0-1)
# ============================

discriminator_mnist = nn.Sequential(
    # Layer 1: 784 → 512
    nn.Linear(img_dim, 512),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),

    # Layer 2: 512 → 256
    nn.Linear(512, 256),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),

    # Layer 3: 256 → 1
    nn.Linear(256, 1),
    nn.Sigmoid()               # Probabilitas [0, 1]
)

print("Discriminator MNIST:")
print(discriminator_mnist)
print(f"\nTotal parameter Discriminator: {sum(p.numel() for p in discriminator_mnist.parameters()):,}")

In [ ]:
# ============================
# VERIFIKASI DIMENSI
# ============================

# Test Generator
test_noise = torch.randn(4, latent_dim)  # 4 sampel, 100 noise
with torch.no_grad():
    test_gen_output = generator_mnist(test_noise)

print(f"Generator Input  : {test_noise.shape}")       # (4, 100)
print(f"Generator Output : {test_gen_output.shape}")   # (4, 784)
print(f"Output range     : [{test_gen_output.min():.3f}, {test_gen_output.max():.3f}]")

# Test Discriminator
test_real = torch.randn(4, img_dim)  # 4 "gambar" acak
with torch.no_grad():
    test_disc_output = discriminator_mnist(test_real)

print(f"\nDiscriminator Input  : {test_real.shape}")        # (4, 784)
print(f"Discriminator Output : {test_disc_output.shape}")   # (4, 1)
print(f"Output range         : [{test_disc_output.min():.3f}, {test_disc_output.max():.3f}]")

In [ ]:
# ============================
# TRAINING GAN MNIST
# ============================

# Pindahkan model ke device
generator_mnist = generator_mnist.to(device)
discriminator_mnist = discriminator_mnist.to(device)

# Loss & Optimizer
criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator_mnist.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator_mnist.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Noise tetap untuk visualisasi (agar bisa melihat perkembangan gambar yang sama)
fixed_noise = torch.randn(64, latent_dim).to(device)

num_epochs = 50
d_losses = []
g_losses = []
gen_images_snapshots = {}

print("Mulai Training Vanilla GAN MNIST...")
print("=" * 65)
start_time = time.time()

for epoch in range(num_epochs):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0
    num_batches = 0

    for real_images, _ in train_loader:
        batch_size = real_images.size(0)
        real_images = real_images.view(batch_size, -1).to(device)  # Flatten: (B,1,28,28)→(B,784)

        # Label
        real_label = torch.ones(batch_size, 1).to(device)
        fake_label = torch.zeros(batch_size, 1).to(device)

        # =============================
        # LANGKAH 1: LATIH DISCRIMINATOR
        # =============================

        # Loss pada data ASLI
        real_output = discriminator_mnist(real_images)
        d_loss_real = criterion(real_output, real_label)

        # Loss pada data PALSU
        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator_mnist(noise)
        fake_output = discriminator_mnist(fake_images.detach())
        d_loss_fake = criterion(fake_output, fake_label)

        # Total loss & update D
        d_loss = d_loss_real + d_loss_fake
        optimizer_D.zero_grad()
        d_loss.backward()
        optimizer_D.step()

        # =============================
        # LANGKAH 2: LATIH GENERATOR
        # =============================

        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator_mnist(noise)
        fake_output = discriminator_mnist(fake_images)
        g_loss = criterion(fake_output, real_label)

        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()

        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()
        num_batches += 1

    avg_d_loss = epoch_d_loss / num_batches
    avg_g_loss = epoch_g_loss / num_batches
    d_losses.append(avg_d_loss)
    g_losses.append(avg_g_loss)

    # Simpan snapshot gambar
    if (epoch + 1) in [1, 5, 10, 20, 30, 50]:
        generator_mnist.eval()
        with torch.no_grad():
            gen_imgs = generator_mnist(fixed_noise).cpu().view(-1, 1, 28, 28)
            gen_images_snapshots[epoch + 1] = gen_imgs
        generator_mnist.train()

    # Print progress
    if (epoch + 1) % 5 == 0:
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
              f"D Loss: {avg_d_loss:.4f} | G Loss: {avg_g_loss:.4f} | "
              f"Time: {elapsed:.1f}s")

print("=" * 65)
print(f"Training Selesai! Total waktu: {time.time() - start_time:.1f}s")

In [ ]:
# ============================
# PLOT TRAINING LOSS
# ============================

plt.figure(figsize=(12, 5))
plt.plot(d_losses, label='Discriminator Loss', color='steelblue', linewidth=2)
plt.plot(g_losses, label='Generator Loss', color='coral', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss — Vanilla GAN MNIST', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# EVOLUSI GAMBAR YANG DIHASILKAN GENERATOR
# ============================

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (ep, images) in enumerate(gen_images_snapshots.items()):
    # Buat grid gambar 8×8
    grid = make_grid(images[:64], nrow=8, normalize=True, padding=2)
    axes[idx].imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    axes[idx].set_title(f'Epoch {ep}', fontsize=14, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Evolusi Gambar Generator Selama Training',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# GENERATE GAMBAR BARU
# ============================

generator_mnist.eval()  # Mode evaluasi

# Generate 64 gambar baru
with torch.no_grad():
    new_noise = torch.randn(64, latent_dim).to(device)
    generated = generator_mnist(new_noise).cpu().view(-1, 1, 28, 28)

# Visualisasi
fig, axes = plt.subplots(8, 8, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    # Denormalisasi dari [-1,1] ke [0,1]
    img = generated[i].squeeze() * 0.5 + 0.5
    ax.imshow(img.numpy(), cmap='gray')
    ax.axis('off')

plt.suptitle('64 Gambar Baru dari Vanilla GAN',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# DATASET MNIST (UNTUK DCGAN)
# ============================

torch.manual_seed(42)

# Transform: resize ke 32×32, normalisasi ke [-1, 1]
# Kita resize ke 32×32 agar dimensi lebih mudah dihitung (kelipatan 2)
transform_dcgan = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset_dc = datasets.MNIST(root='./data', train=True,
                                    download=True, transform=transform_dcgan)

batch_size = 128
train_loader_dc = DataLoader(train_dataset_dc, batch_size=batch_size, shuffle=True)

print(f"Training samples : {len(train_dataset_dc)}")
print(f"Image shape      : {train_dataset_dc[0][0].shape}")  # (1, 32, 32)
print(f"Batch size       : {batch_size}")

In [ ]:
# ============================
# GENERATOR DCGAN
# ============================

# Arsitektur Generator DCGAN:
#   Noise (100) → Reshape → ConvTranspose2d layers → Gambar (1, 32, 32)
#
#   z (100,)
#     ↓ Linear + Reshape
#   (256, 4, 4)
#     ↓ ConvTranspose2d (stride=2, padding=1)
#   (128, 8, 8)
#     ↓ ConvTranspose2d (stride=2, padding=1)
#   (64, 16, 16)
#     ↓ ConvTranspose2d (stride=2, padding=1)
#   (1, 32, 32) — gambar output

latent_dim = 100

generator_dcgan = nn.Sequential(
    # Langkah 1: Noise → (256 × 4 × 4)
    nn.Linear(latent_dim, 256 * 4 * 4),
    nn.BatchNorm1d(256 * 4 * 4),
    nn.ReLU(),

    # Reshape: vektor → tensor 3D (menggunakan nn.Unflatten)
    nn.Unflatten(1, (256, 4, 4)),   # (batch, 256*4*4) → (batch, 256, 4, 4)

    # Langkah 2: (256, 4, 4) → (128, 8, 8)
    nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),

    # Langkah 3: (128, 8, 8) → (64, 16, 16)
    nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),

    # Langkah 4: (64, 16, 16) → (1, 32, 32) — Output!
    nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),
    nn.Tanh()                    # Output: [-1, 1]
)

print("Generator DCGAN:")
print(generator_dcgan)
print(f"\nTotal parameter: {sum(p.numel() for p in generator_dcgan.parameters()):,}")

In [ ]:
# ============================
# DISCRIMINATOR DCGAN
# ============================

# Arsitektur Discriminator DCGAN:
#   Gambar (1, 32, 32) → Conv2d layers → Probabilitas (0-1)
#
#   (1, 32, 32)
#     ↓ Conv2d (stride=2, padding=1)
#   (64, 16, 16)
#     ↓ Conv2d (stride=2, padding=1)
#   (128, 8, 8)
#     ↓ Conv2d (stride=2, padding=1)
#   (256, 4, 4)
#     ↓ Flatten → Linear → Sigmoid
#   P(real)

discriminator_dcgan = nn.Sequential(
    # Langkah 1: (1, 32, 32) → (64, 16, 16)
    nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),
    nn.LeakyReLU(0.2),
    # Catatan: TIDAK ada BatchNorm pada layer pertama Discriminator (sesuai paper DCGAN)

    # Langkah 2: (64, 16, 16) → (128, 8, 8)
    nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
    nn.BatchNorm2d(128),
    nn.LeakyReLU(0.2),

    # Langkah 3: (128, 8, 8) → (256, 4, 4)
    nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
    nn.BatchNorm2d(256),
    nn.LeakyReLU(0.2),

    # Langkah 4: Flatten → FC → Sigmoid
    nn.Flatten(),                    # (256, 4, 4) → (4096)
    nn.Linear(256 * 4 * 4, 1),
    nn.Sigmoid()                     # Probabilitas [0, 1]
)

print("Discriminator DCGAN:")
print(discriminator_dcgan)
print(f"\nTotal parameter: {sum(p.numel() for p in discriminator_dcgan.parameters()):,}")

In [ ]:
# ============================
# VERIFIKASI DIMENSI DCGAN
# ============================

# Test Generator
test_noise = torch.randn(4, latent_dim)
with torch.no_grad():
    test_gen = generator_dcgan(test_noise)

print(f"Generator DCGAN:")
print(f"  Input  : {test_noise.shape}")       # (4, 100)
print(f"  Output : {test_gen.shape}")          # (4, 1, 32, 32)
print(f"  Range  : [{test_gen.min():.3f}, {test_gen.max():.3f}]")

# Test Discriminator
test_img = torch.randn(4, 1, 32, 32)
with torch.no_grad():
    test_disc = discriminator_dcgan(test_img)

print(f"\nDiscriminator DCGAN:")
print(f"  Input  : {test_img.shape}")          # (4, 1, 32, 32)
print(f"  Output : {test_disc.shape}")         # (4, 1)
print(f"  Range  : [{test_disc.min():.3f}, {test_disc.max():.3f}]")

# Verifikasi layer per layer Generator
print("\n--- Verifikasi Layer Generator ---")
x = torch.randn(2, latent_dim)
print(f"  Input              : {x.shape}")
x = nn.Linear(100, 256*4*4)(x);       print(f"  Linear(100→4096)   : {x.shape}")
x = nn.Unflatten(1, (256,4,4))(x);    print(f"  Unflatten          : {x.shape}")
x = nn.ConvTranspose2d(256,128,4,2,1)(x); print(f"  ConvT(256→128)     : {x.shape}")
x = nn.ConvTranspose2d(128,64,4,2,1)(x);  print(f"  ConvT(128→64)      : {x.shape}")
x = nn.ConvTranspose2d(64,1,4,2,1)(x);    print(f"  ConvT(64→1)        : {x.shape}")

In [ ]:
# ============================
# INISIALISASI BOBOT (SESUAI PAPER DCGAN)
# ============================

# Paper DCGAN merekomendasikan inisialisasi bobot dari N(0, 0.02)
# Kita lakukan dengan iterasi sederhana tanpa def()

for layer in generator_dcgan.modules():
    if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(layer.weight.data, 0.0, 0.02)
    if isinstance(layer, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.normal_(layer.weight.data, 1.0, 0.02)
        nn.init.constant_(layer.bias.data, 0)

for layer in discriminator_dcgan.modules():
    if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(layer.weight.data, 0.0, 0.02)
    if isinstance(layer, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.normal_(layer.weight.data, 1.0, 0.02)
        nn.init.constant_(layer.bias.data, 0)

print("✅ Bobot Generator dan Discriminator berhasil diinisialisasi!")
print("   Metode: Normal(0, 0.02) — sesuai paper DCGAN")

In [ ]:
# ============================
# TRAINING DCGAN
# ============================

# Pindahkan ke device
generator_dcgan = generator_dcgan.to(device)
discriminator_dcgan = discriminator_dcgan.to(device)

# Loss & Optimizer
criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator_dcgan.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator_dcgan.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Fixed noise untuk tracking perkembangan
fixed_noise = torch.randn(64, latent_dim).to(device)

num_epochs = 30
d_losses = []
g_losses = []
dc_gen_snapshots = {}

print("Mulai Training DCGAN MNIST...")
print("=" * 65)
start_time = time.time()

for epoch in range(num_epochs):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0
    num_batches = 0

    for real_images, _ in train_loader_dc:
        batch_size = real_images.size(0)
        real_images = real_images.to(device)  # (B, 1, 32, 32) — TIDAK perlu flatten!

        real_label = torch.ones(batch_size, 1).to(device)
        fake_label = torch.zeros(batch_size, 1).to(device)

        # =============================
        # LANGKAH 1: LATIH DISCRIMINATOR
        # =============================

        real_output = discriminator_dcgan(real_images)
        d_loss_real = criterion(real_output, real_label)

        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator_dcgan(noise)
        fake_output = discriminator_dcgan(fake_images.detach())
        d_loss_fake = criterion(fake_output, fake_label)

        d_loss = d_loss_real + d_loss_fake
        optimizer_D.zero_grad()
        d_loss.backward()
        optimizer_D.step()

        # =============================
        # LANGKAH 2: LATIH GENERATOR
        # =============================

        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator_dcgan(noise)
        fake_output = discriminator_dcgan(fake_images)
        g_loss = criterion(fake_output, real_label)

        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()

        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()
        num_batches += 1

    avg_d_loss = epoch_d_loss / num_batches
    avg_g_loss = epoch_g_loss / num_batches
    d_losses.append(avg_d_loss)
    g_losses.append(avg_g_loss)

    # Simpan snapshot
    if (epoch + 1) in [1, 5, 10, 15, 20, 30]:
        generator_dcgan.eval()
        with torch.no_grad():
            gen_imgs = generator_dcgan(fixed_noise).cpu()
            dc_gen_snapshots[epoch + 1] = gen_imgs
        generator_dcgan.train()

    # Print progress
    if (epoch + 1) % 5 == 0:
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
              f"D Loss: {avg_d_loss:.4f} | G Loss: {avg_g_loss:.4f} | "
              f"Time: {elapsed:.1f}s")

print("=" * 65)
print(f"Training Selesai! Total waktu: {time.time() - start_time:.1f}s")

In [ ]:
# ============================
# PLOT 1: TRAINING LOSS
# ============================

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Loss
axes[0].plot(d_losses, label='Discriminator Loss', color='steelblue', linewidth=2)
axes[0].plot(g_losses, label='Generator Loss', color='coral', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss — DCGAN MNIST', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Generate gambar akhir
generator_dcgan.eval()
with torch.no_grad():
    final_images = generator_dcgan(fixed_noise[:16]).cpu()

grid = make_grid(final_images, nrow=4, normalize=True, padding=2)
axes[1].imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
axes[1].set_title('Gambar Generated (Akhir Training)', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ============================
# PLOT 2: EVOLUSI GAMBAR DCGAN
# ============================

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (ep, images) in enumerate(dc_gen_snapshots.items()):
    grid = make_grid(images[:64], nrow=8, normalize=True, padding=2)
    axes[idx].imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    axes[idx].set_title(f'Epoch {ep}', fontsize=14, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Evolusi DCGAN — Gambar MNIST',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# INTERPOLASI DI LATENT SPACE
# ============================

generator_dcgan.eval()

# Pilih 2 titik acak di latent space
z1 = torch.randn(1, latent_dim).to(device)
z2 = torch.randn(1, latent_dim).to(device)

# Interpolasi linear antara z1 dan z2
n_steps = 10
interpolated = []

for alpha in np.linspace(0, 1, n_steps):
    z_interp = (1 - alpha) * z1 + alpha * z2
    interpolated.append(z_interp)

z_batch = torch.cat(interpolated, dim=0)

with torch.no_grad():
    interp_images = generator_dcgan(z_batch).cpu()

# Visualisasi
fig, axes = plt.subplots(1, n_steps, figsize=(20, 3))
for i in range(n_steps):
    img = interp_images[i].squeeze() * 0.5 + 0.5
    axes[i].imshow(img.numpy(), cmap='gray')
    axes[i].set_title(f'α={i/(n_steps-1):.1f}', fontsize=9)
    axes[i].axis('off')

plt.suptitle('Interpolasi di Latent Space — Transisi Halus Antar Digit',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# DATASET FASHION MNIST
# ============================

torch.manual_seed(42)

# Class names Fashion MNIST
fashion_classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                   'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Transform: resize ke 32×32, normalisasi ke [-1, 1]
transform_fashion = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset_fashion = datasets.FashionMNIST(root='./data', train=True,
                                                download=True,
                                                transform=transform_fashion)

batch_size = 128
train_loader_fashion = DataLoader(train_dataset_fashion, batch_size=batch_size,
                                   shuffle=True)

print(f"Training samples : {len(train_dataset_fashion)}")
print(f"Image shape      : {train_dataset_fashion[0][0].shape}")
print(f"Classes          : {fashion_classes}")

# Visualisasi sampel
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset_fashion[i * 600]
    ax.imshow(image.squeeze() * 0.5 + 0.5, cmap='gray')
    ax.set_title(fashion_classes[label], fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sampel Fashion MNIST', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# GENERATOR DCGAN — FASHION MNIST
# ============================

latent_dim = 100

generator_fashion = nn.Sequential(
    # Noise → Tensor 3D
    nn.Linear(latent_dim, 256 * 4 * 4),
    nn.BatchNorm1d(256 * 4 * 4),
    nn.ReLU(),
    nn.Unflatten(1, (256, 4, 4)),

    # (256, 4, 4) → (128, 8, 8)
    nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),

    # (128, 8, 8) → (64, 16, 16)
    nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),

    # (64, 16, 16) → (1, 32, 32)
    nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),
    nn.Tanh()
)

# ============================
# DISCRIMINATOR DCGAN — FASHION MNIST
# ============================

discriminator_fashion = nn.Sequential(
    # (1, 32, 32) → (64, 16, 16)
    nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),
    nn.LeakyReLU(0.2),

    # (64, 16, 16) → (128, 8, 8)
    nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
    nn.BatchNorm2d(128),
    nn.LeakyReLU(0.2),

    # (128, 8, 8) → (256, 4, 4)
    nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
    nn.BatchNorm2d(256),
    nn.LeakyReLU(0.2),

    # Flatten → FC → Sigmoid
    nn.Flatten(),
    nn.Linear(256 * 4 * 4, 1),
    nn.Sigmoid()
)

print(f"Generator parameter     : {sum(p.numel() for p in generator_fashion.parameters()):,}")
print(f"Discriminator parameter : {sum(p.numel() for p in discriminator_fashion.parameters()):,}")

In [ ]:
# ============================
# INISIALISASI BOBOT
# ============================

for layer in generator_fashion.modules():
    if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(layer.weight.data, 0.0, 0.02)
    if isinstance(layer, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.normal_(layer.weight.data, 1.0, 0.02)
        nn.init.constant_(layer.bias.data, 0)

for layer in discriminator_fashion.modules():
    if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(layer.weight.data, 0.0, 0.02)
    if isinstance(layer, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.normal_(layer.weight.data, 1.0, 0.02)
        nn.init.constant_(layer.bias.data, 0)

print("Bobot berhasil diinisialisasi!")

# ============================
# TRAINING DCGAN FASHION MNIST
# ============================

generator_fashion = generator_fashion.to(device)
discriminator_fashion = discriminator_fashion.to(device)

criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator_fashion.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator_fashion.parameters(), lr=0.0002, betas=(0.5, 0.999))

fixed_noise = torch.randn(64, latent_dim).to(device)

num_epochs = 50
d_losses_f = []
g_losses_f = []
fashion_snapshots = {}

print("\nMulai Training DCGAN Fashion MNIST...")
print("=" * 65)
start_time = time.time()

for epoch in range(num_epochs):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0
    num_batches = 0

    for real_images, _ in train_loader_fashion:
        batch_size = real_images.size(0)
        real_images = real_images.to(device)

        real_label = torch.ones(batch_size, 1).to(device)
        fake_label = torch.zeros(batch_size, 1).to(device)

        # --- Latih Discriminator ---
        real_output = discriminator_fashion(real_images)
        d_loss_real = criterion(real_output, real_label)

        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator_fashion(noise)
        fake_output = discriminator_fashion(fake_images.detach())
        d_loss_fake = criterion(fake_output, fake_label)

        d_loss = d_loss_real + d_loss_fake
        optimizer_D.zero_grad()
        d_loss.backward()
        optimizer_D.step()

        # --- Latih Generator ---
        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator_fashion(noise)
        fake_output = discriminator_fashion(fake_images)
        g_loss = criterion(fake_output, real_label)

        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()

        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()
        num_batches += 1

    avg_d_loss = epoch_d_loss / num_batches
    avg_g_loss = epoch_g_loss / num_batches
    d_losses_f.append(avg_d_loss)
    g_losses_f.append(avg_g_loss)

    # Simpan snapshot
    if (epoch + 1) in [1, 10, 20, 30, 40, 50]:
        generator_fashion.eval()
        with torch.no_grad():
            gen_imgs = generator_fashion(fixed_noise).cpu()
            fashion_snapshots[epoch + 1] = gen_imgs
        generator_fashion.train()

    if (epoch + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
              f"D Loss: {avg_d_loss:.4f} | G Loss: {avg_g_loss:.4f} | "
              f"Time: {elapsed:.1f}s")

print("=" * 65)
print(f"Training Selesai! Total waktu: {time.time() - start_time:.1f}s")

In [ ]:
# ============================
# EVOLUSI GAMBAR FASHION MNIST
# ============================

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (ep, images) in enumerate(fashion_snapshots.items()):
    grid = make_grid(images[:64], nrow=8, normalize=True, padding=2)
    axes[idx].imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    axes[idx].set_title(f'Epoch {ep}', fontsize=14, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Evolusi DCGAN — Fashion MNIST',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# PERBANDINGAN: ASLI vs GENERATED
# ============================

fig, axes = plt.subplots(2, 8, figsize=(18, 5))

# Baris atas: Data ASLI
for i in range(8):
    image, label = train_dataset_fashion[i * 700]
    axes[0, i].imshow(image.squeeze() * 0.5 + 0.5, cmap='gray')
    axes[0, i].set_title(fashion_classes[label], fontsize=9)
    axes[0, i].axis('off')
axes[0, 0].set_ylabel('ASLI', fontsize=12, fontweight='bold', rotation=0, labelpad=40)

# Baris bawah: Data GENERATED
generator_fashion.eval()
with torch.no_grad():
    gen_noise = torch.randn(8, latent_dim).to(device)
    gen_images = generator_fashion(gen_noise).cpu()

for i in range(8):
    axes[1, i].imshow(gen_images[i].squeeze() * 0.5 + 0.5, cmap='gray')
    axes[1, i].set_title('Generated', fontsize=9)
    axes[1, i].axis('off')
axes[1, 0].set_ylabel('GENERATED', fontsize=12, fontweight='bold', rotation=0, labelpad=60)

plt.suptitle('Data Asli vs Generated — Fashion MNIST',
             fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# MODUL PEMBUNGKUS UNTUK cGAN
# ============================

# Untuk cGAN, kita meng-embed label ke vektor dan menggabungkannya
# dengan noise (Generator) atau gambar (Discriminator)

# Label embedding: angka label → vektor representasi
# Contoh: label "3" → [0.2, -0.5, 1.1, ...] (vektor 10-dim)

# Kita akan menggunakan nn.Embedding di dalam nn.Sequential
# dengan cara menggabungkan noise+label SEBELUM masuk ke model

n_classes = 10    # 10 digit (0-9)
embed_dim = 10    # Dimensi embedding label
latent_dim = 100  # Dimensi noise

# Label embedding layer (dipake untuk Generator DAN Discriminator)
label_emb_gen = nn.Embedding(n_classes, embed_dim)
label_emb_disc = nn.Embedding(n_classes, embed_dim)

print(f"Label Embedding:")
print(f"  Input : integer 0-9")
print(f"  Output: vektor {embed_dim}-dim")
print(f"  Contoh: label=3 → {label_emb_gen(torch.tensor([3])).detach()}")

In [ ]:
# ============================
# GENERATOR cGAN
# ============================

# Input Generator cGAN: [noise (100) concat label_embedding (10)] = 110

generator_cgan = nn.Sequential(
    # Layer 1: 110 → 256
    nn.Linear(latent_dim + embed_dim, 256),
    nn.BatchNorm1d(256),
    nn.LeakyReLU(0.2),

    # Layer 2: 256 → 512
    nn.Linear(256, 512),
    nn.BatchNorm1d(512),
    nn.LeakyReLU(0.2),

    # Layer 3: 512 → 1024
    nn.Linear(512, 1024),
    nn.BatchNorm1d(1024),
    nn.LeakyReLU(0.2),

    # Layer 4: 1024 → 784 (28×28)
    nn.Linear(1024, 28 * 28),
    nn.Tanh()
)

print("Generator cGAN:")
print(generator_cgan)
print(f"Total parameter: {sum(p.numel() for p in generator_cgan.parameters()):,}")

In [ ]:
# ============================
# DISCRIMINATOR cGAN
# ============================

# Input Discriminator cGAN: [gambar (784) concat label_embedding (10)] = 794

discriminator_cgan = nn.Sequential(
    # Layer 1: 794 → 512
    nn.Linear(28 * 28 + embed_dim, 512),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),

    # Layer 2: 512 → 256
    nn.Linear(512, 256),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),

    # Layer 3: 256 → 1
    nn.Linear(256, 1),
    nn.Sigmoid()
)

print("Discriminator cGAN:")
print(discriminator_cgan)
print(f"Total parameter: {sum(p.numel() for p in discriminator_cgan.parameters()):,}")

In [ ]:
# ============================
# DATASET MNIST UNTUK cGAN
# ============================

transform_cgan = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset_cgan = datasets.MNIST(root='./data', train=True,
                                      download=True, transform=transform_cgan)

batch_size = 128
train_loader_cgan = DataLoader(train_dataset_cgan, batch_size=batch_size, shuffle=True)

# ============================
# TRAINING cGAN
# ============================

# Pindahkan ke device
generator_cgan = generator_cgan.to(device)
discriminator_cgan = discriminator_cgan.to(device)
label_emb_gen = label_emb_gen.to(device)
label_emb_disc = label_emb_disc.to(device)

criterion = nn.BCELoss()

# Optimizer untuk Generator + embedding Generator
optimizer_G = optim.Adam(
    list(generator_cgan.parameters()) + list(label_emb_gen.parameters()),
    lr=0.0002, betas=(0.5, 0.999)
)

# Optimizer untuk Discriminator + embedding Discriminator
optimizer_D = optim.Adam(
    list(discriminator_cgan.parameters()) + list(label_emb_disc.parameters()),
    lr=0.0002, betas=(0.5, 0.999)
)

# Fixed noise & labels untuk visualisasi
fixed_noise = torch.randn(100, latent_dim).to(device)
# 10 gambar per digit: 0,0,...,0, 1,1,...,1, ..., 9,9,...,9
fixed_labels = torch.arange(10).repeat_interleave(10).to(device)

num_epochs = 50
d_losses_c = []
g_losses_c = []
cgan_snapshots = {}

print("Mulai Training Conditional GAN...")
print("=" * 65)
start_time = time.time()

for epoch in range(num_epochs):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0
    num_batches = 0

    for real_images, labels in train_loader_cgan:
        batch_size = real_images.size(0)
        real_images = real_images.view(batch_size, -1).to(device)
        labels = labels.to(device)

        real_label = torch.ones(batch_size, 1).to(device)
        fake_label = torch.zeros(batch_size, 1).to(device)

        # =============================
        # LANGKAH 1: LATIH DISCRIMINATOR
        # =============================

        # Embed label untuk Discriminator
        label_embedded_d = label_emb_disc(labels)

        # Data ASLI + label → D
        real_input_d = torch.cat([real_images, label_embedded_d], dim=1)
        real_output = discriminator_cgan(real_input_d)
        d_loss_real = criterion(real_output, real_label)

        # Data PALSU + label → D
        noise = torch.randn(batch_size, latent_dim).to(device)
        label_embedded_g = label_emb_gen(labels)
        gen_input = torch.cat([noise, label_embedded_g], dim=1)
        fake_images = generator_cgan(gen_input)

        fake_input_d = torch.cat([fake_images.detach(), label_embedded_d], dim=1)
        fake_output = discriminator_cgan(fake_input_d)
        d_loss_fake = criterion(fake_output, fake_label)

        d_loss = d_loss_real + d_loss_fake
        optimizer_D.zero_grad()
        d_loss.backward()
        optimizer_D.step()

        # =============================
        # LANGKAH 2: LATIH GENERATOR
        # =============================

        noise = torch.randn(batch_size, latent_dim).to(device)
        label_embedded_g = label_emb_gen(labels)
        gen_input = torch.cat([noise, label_embedded_g], dim=1)
        fake_images = generator_cgan(gen_input)

        label_embedded_d = label_emb_disc(labels)
        fake_input_d = torch.cat([fake_images, label_embedded_d], dim=1)
        fake_output = discriminator_cgan(fake_input_d)
        g_loss = criterion(fake_output, real_label)

        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()

        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()
        num_batches += 1

    avg_d_loss = epoch_d_loss / num_batches
    avg_g_loss = epoch_g_loss / num_batches
    d_losses_c.append(avg_d_loss)
    g_losses_c.append(avg_g_loss)

    # Simpan snapshot
    if (epoch + 1) in [1, 10, 20, 30, 40, 50]:
        generator_cgan.eval()
        with torch.no_grad():
            fixed_emb = label_emb_gen(fixed_labels)
            fixed_input = torch.cat([fixed_noise, fixed_emb], dim=1)
            gen_imgs = generator_cgan(fixed_input).cpu().view(-1, 1, 28, 28)
            cgan_snapshots[epoch + 1] = gen_imgs
        generator_cgan.train()

    if (epoch + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
              f"D Loss: {avg_d_loss:.4f} | G Loss: {avg_g_loss:.4f} | "
              f"Time: {elapsed:.1f}s")

print("=" * 65)
print(f"Training Selesai! Total waktu: {time.time() - start_time:.1f}s")

In [ ]:
# ============================
# EVOLUSI GAMBAR cGAN
# ============================

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (ep, images) in enumerate(cgan_snapshots.items()):
    # Grid 10 baris (digit 0-9), 10 kolom (variasi per digit)
    grid = make_grid(images[:100], nrow=10, normalize=True, padding=2)
    axes[idx].imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    axes[idx].set_title(f'Epoch {ep}', fontsize=14, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Evolusi cGAN — Setiap Baris = 1 Digit (0-9)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# GENERATE DIGIT TERTENTU
# ============================

generator_cgan.eval()

# Generate 8 variasi untuk setiap digit (0-9)
fig, axes = plt.subplots(10, 8, figsize=(14, 18))

for digit in range(10):
    with torch.no_grad():
        noise = torch.randn(8, latent_dim).to(device)
        labels = torch.full((8,), digit, dtype=torch.long).to(device)
        emb = label_emb_gen(labels)
        gen_input = torch.cat([noise, emb], dim=1)
        generated = generator_cgan(gen_input).cpu().view(-1, 1, 28, 28)

    for j in range(8):
        img = generated[j].squeeze() * 0.5 + 0.5
        axes[digit, j].imshow(img.numpy(), cmap='gray')
        axes[digit, j].axis('off')

    axes[digit, 0].set_ylabel(f'Digit {digit}', fontsize=12,
                               fontweight='bold', rotation=0, labelpad=40)

plt.suptitle('Conditional GAN — Generate Digit Tertentu\n'
             '(Setiap baris = 1 digit, 8 variasi)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# INTERPOLASI ANTAR DIGIT DI LATENT SPACE
# ============================

generator_cgan.eval()

# Interpolasi di latent space untuk DIGIT YANG SAMA (digit 5)
z1 = torch.randn(1, latent_dim).to(device)
z2 = torch.randn(1, latent_dim).to(device)
n_steps = 10

fig, axes = plt.subplots(2, n_steps, figsize=(20, 5))

# Baris 1: Interpolasi noise DENGAN DIGIT SAMA (digit 5)
for i, alpha in enumerate(np.linspace(0, 1, n_steps)):
    with torch.no_grad():
        z_interp = ((1 - alpha) * z1 + alpha * z2)
        label = torch.tensor([5]).to(device)
        emb = label_emb_gen(label)
        gen_input = torch.cat([z_interp, emb], dim=1)
        img = generator_cgan(gen_input).cpu().view(28, 28) * 0.5 + 0.5

    axes[0, i].imshow(img.numpy(), cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title(f'α={alpha:.1f}', fontsize=9)

axes[0, 0].set_ylabel('Digit 5\n(beda gaya)', fontsize=11,
                        fontweight='bold', rotation=0, labelpad=60)

# Baris 2: NOISE TETAP, DIGIT BERUBAH (0 → 9)
fixed_z = torch.randn(1, latent_dim).to(device)
for i in range(n_steps):
    with torch.no_grad():
        label = torch.tensor([i]).to(device)
        emb = label_emb_gen(label)
        gen_input = torch.cat([fixed_z, emb], dim=1)
        img = generator_cgan(gen_input).cpu().view(28, 28) * 0.5 + 0.5

    axes[1, i].imshow(img.numpy(), cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title(f'Digit {i}', fontsize=9)

axes[1, 0].set_ylabel('Noise tetap\n(beda digit)', fontsize=11,
                        fontweight='bold', rotation=0, labelpad=60)

plt.suptitle('Eksplorasi Conditional GAN\n'
             'Atas: sama digit, beda gaya | Bawah: beda digit, sama gaya',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()